# Libraries

In [1]:
import math
from datasets import load_dataset
from fastapi import FastAPI
from huggingface_hub import notebook_login
from pydantic import BaseModel
from transformers import (AutoModelForCausalLM, AutoTokenizer,
                          Trainer, TrainingArguments)

# Authentication

In [2]:
notebook_login()

# Load Dataset and View its Structure

In [3]:
ds = load_dataset("rajpurkar/squad")

# Dataset Structure
print(ds)
print("\nTraining set:", ds["train"][0])
print("\nTesting set:", ds["validation"][0])

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

plain_text/train-00000-of-00001.parquet:   0%|          | 0.00/14.5M [00:00<?, ?B/s]

plain_text/validation-00000-of-00001.par(…):   0%|          | 0.00/1.82M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/87599 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/10570 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['id', 'title', 'context', 'question', 'answers'],
        num_rows: 87599
    })
    validation: Dataset({
        features: ['id', 'title', 'context', 'question', 'answers'],
        num_rows: 10570
    })
})

Training set: {'id': '5733be284776f41900661182', 'title': 'University_of_Notre_Dame', 'context': 'Architecturally, the school has a Catholic character. Atop the Main Building\'s gold dome is a golden statue of the Virgin Mary. Immediately in front of the Main Building and facing it, is a copper statue of Christ with arms upraised with the legend "Venite Ad Me Omnes". Next to the Main Building is the Basilica of the Sacred Heart. Immediately behind the basilica is the Grotto, a Marian place of prayer and reflection. It is a replica of the grotto at Lourdes, France where the Virgin Mary reputedly appeared to Saint Bernadette Soubirous in 1858. At the end of the main drive (and in a direct line that connects through 3 statues an

# Data Preprocessing

In [4]:
tokenizer = AutoTokenizer.from_pretrained("openai-community/gpt2")

special_tokens = tokenizer.special_tokens_map
print(special_tokens)

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

{'bos_token': '<|endoftext|>', 'eos_token': '<|endoftext|>', 'unk_token': '<|endoftext|>'}


In [5]:
def append_bos_to_question(input_dict: dict) -> dict:
  """
  :param input_dict: The data type of the input data is dictionary.
  :return: The input data after processing.
  """
  input_dict['question'] += special_tokens['bos_token']
  return input_dict


ds = ds.remove_columns(['id', 'title', 'context', 'answers'])
ds = ds.map(append_bos_to_question)

Map:   0%|          | 0/87599 [00:00<?, ? examples/s]

Map:   0%|          | 0/10570 [00:00<?, ? examples/s]

# Tokenization

In [6]:
def tokenize_func(input_dict: dict):
    return tokenizer(input_dict['question'], truncation=True)


tokenized_ds = ds.map(tokenize_func, batched=True, num_proc=4,
                      remove_columns=['question'])
tokenized_ds

Map (num_proc=4):   0%|          | 0/87599 [00:00<?, ? examples/s]

Map (num_proc=4):   0%|          | 0/10570 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask'],
        num_rows: 87599
    })
    validation: Dataset({
        features: ['input_ids', 'attention_mask'],
        num_rows: 10570
    })
})

In [7]:
max_block_len = 256


def divide_tokenized_text(tokenized_text_dict: dict, block_size: int) -> dict:
    """
    Divide the tokenized text in the examples into fixed-length blocks with the
    size block_size to efficiently process large datasets during natural
    language processing tasks.

    :param tokenized_text_dict: The tokenized text as values for different keys.
    :param block_size: The desired length of each tokenized block.
    :return: A dictionary with tokenized text divided into fixed-length blocks.
    """
    concat_emp = {k: sum(tokenized_text_dict[k],
     []) for k in tokenized_text_dict.keys()}
    total_len = len(concat_emp[list(tokenized_text_dict.keys())[0]])
    total_len = (total_len // block_size) * block_size

    res = {k: [t[i:i+block_size] for i in range(0, total_len,
                                                block_size)] for k,
           t in concat_emp.items()}

    res['labels'] = res['input_ids'].copy()
    return res


processed_ds = tokenized_ds.map(
    lambda tokenized_text_dict: divide_tokenized_text(tokenized_text_dict,
                                                      max_block_len),
    batched=True,
    batch_size=1000,
    num_proc=4
)

Map (num_proc=4):   0%|          | 0/87599 [00:00<?, ? examples/s]

Map (num_proc=4):   0%|          | 0/10570 [00:00<?, ? examples/s]

# Split the Dataset

In [8]:
train_ds = processed_ds['train'].shuffle(seed=42)
eval_ds = processed_ds['validation'].shuffle(seed=42)

# Model Training

In [9]:
# Define the model
model = AutoModelForCausalLM.from_pretrained("openai-community/gpt2")
tokenizer.add_special_tokens({'pad_token': '[PAD]'})

training_args = TrainingArguments(
    './GPT2-SQuAD',
    eval_strategy="epoch",
    num_train_epochs=5,
    learning_rate=2e-5,
    logging_steps=50,
    logging_dir="./train_logs",
    weight_decay=0.01,
    push_to_hub=False)

trainer = Trainer(model=model, args=training_args, train_dataset=train_ds,
                  eval_dataset=eval_ds, tokenizer=tokenizer)

trainer.train()

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

/tmp/ipython-input-3202586011.py:15: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(model=model, args=training_args, train_dataset=train_ds,
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 50257}.
/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:

 2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Find your API key here: https://wandb.ai/authorize?ref=models
wandb: Paste an API key from your profile and hit enter:

 ··········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: 2755542479 (2755542479-columbia-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Epoch,Training Loss,Validation Loss
1,2.899700,2.826546
2,2.840900,2.817265
3,2.765800,2.811338
4,2.753800,2.811040
5,2.747600,2.811368


TrainOutput(global_step=2840, training_loss=2.820360820394167, metrics={'train_runtime': 1793.0205, 'train_samples_per_second': 12.669, 'train_steps_per_second': 1.584, 'total_flos': 2967624253440000.0, 'train_loss': 2.820360820394167, 'epoch': 5.0})

# Model Evaluation

In [10]:
eval_results = trainer.evaluate()
print(f'Perplexity: {math.exp(eval_results["eval_loss"])}')

Perplexity: 16.63266387733165


# Push the Model and Tokenizer

In [12]:
model.push_to_hub('GPT2-SQuAD')
tokenizer.push_to_hub("GPT2-SQuAD")

README.md:   0%|          | 0.00/463 [00:00<?, ?B/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  GPT2-SQuAD/model.safetensors:   0%|          |  549kB /  498MB            

CommitInfo(commit_url='https://huggingface.co/Capital-Ink-Morning-Glory/GPT2-SQuAD/commit/46c378ca4d0bd28ef383bc44ad6a741cab75037c', commit_message='Upload tokenizer', commit_description='', oid='46c378ca4d0bd28ef383bc44ad6a741cab75037c', pr_url=None, repo_url=RepoUrl('https://huggingface.co/Capital-Ink-Morning-Glory/GPT2-SQuAD', endpoint='https://huggingface.co', repo_type='model', repo_id='Capital-Ink-Morning-Glory/GPT2-SQuAD'), pr_revision=None, pr_num=None)